# Chargeback & Dispute Intelligence

Build a **grounded, self-verifying** agent that explains a payments processor's **chargeback spikes** — by reason code, network and merchant category — and benchmarks internal disputes against **real CFPB consumer complaints**.

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`DISPUTE_BENCHMARK`) — holiday spike by reason category |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Prereq:** run `setup_synthetic.sql` (no listing) or `setup.sql` (real CFPB complaint benchmark). Set the notebook role to one that owns `SNOWPACK_DISPUTE_INTEL.CORE`.

In [ ]:
SCHEMA = "SNOWPACK_DISPUTE_INTEL.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.DISPUTE_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`DISPUTE_BENCHMARK` = the dispute book pre-aggregated to (merchant category, network, reason code, month), with chargeback rate (bps), win rate, friendly-fraud rate, and a **monthly** CFPB complaint benchmark (non-additive — never sum it across reason codes). Here: disputes and win rate by reason category per month, highlighting the holiday spike.

In [ ]:
session.sql(f"""
  SELECT reason_category, month_end,
         SUM(dispute_count) AS disputes,
         ROUND(SUM(total_dispute_amount)/1e3,1) AS disputed_k,
         ROUND(SUM(won_count)/NULLIF(SUM(resolved_count),0),3) AS win_rate,
         IFF(MONTH(month_end) IN (11,12,1),'holiday','') AS season
  FROM {SCHEMA}.DISPUTE_BENCHMARK
  GROUP BY reason_category, month_end
  ORDER BY month_end DESC, disputes DESC
  LIMIT 20
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`DISPUTE_SV` lets Analyst answer NL questions like *"which reason codes drove disputes in 2025?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English. Overall win rate = `total_won / total_resolved`.

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.DISPUTE_SV
      METRICS total_disputes, total_dispute_usd, total_won, total_resolved
      DIMENSIONS reason_category, report_year)
  WHERE report_year = 2025 ORDER BY total_disputes DESC
""").to_pandas()

## Step 3 — Verified fact pack

Start-of-window and latest snapshots per reason category — the agent's only allowed numbers. CFPB complaints come in via `MAX` (monthly benchmark, non-additive).

In [ ]:
fact_pack = session.sql(f"""
  WITH w AS (SELECT MIN(month_end) a, MAX(month_end) z FROM {SCHEMA}.DISPUTE_BENCHMARK),
  agg AS (
    SELECT s.reason_category, s.month_end,
           SUM(s.dispute_count) AS dispute_count,
           SUM(s.total_dispute_amount) AS dispute_usd,
           SUM(s.won_count) AS won, SUM(s.resolved_count) AS resolved,
           SUM(s.friendly_fraud_count) AS friendly_fraud,
           MAX(s.cfpb_complaint_count) AS cfpb_complaints
    FROM {SCHEMA}.DISPUTE_BENCHMARK s JOIN w ON s.month_end IN (w.a, w.z)
    GROUP BY 1, 2)
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'reason_category', reason_category, 'month', month_end::string,
           'dispute_count', dispute_count, 'dispute_usd', dispute_usd,
           'win_rate', ROUND(won/NULLIF(resolved,0), 3),
           'friendly_fraud_rate', ROUND(friendly_fraud/NULLIF(dispute_count,0), 3),
           'cfpb_complaints', cfpb_complaints))
  FROM agg
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a payments dispute-ops analyst. Write a concise chargeback note for the processor using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; win_rate and friendly_fraud_rate are fractions (0-1), express as %; "
    "dollar fields are USD, express in $M/$K; cfpb_complaints is a monthly external benchmark (do not sum it across reason categories); "
    "compute change as (latest month - earliest month) per reason category from FACTS; no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (which reason categories drove disputes), a table (reason category | disputes start->now | disputed $ | win rate | friendly-fraud rate), then 3 bullets, ending with '## Data Coverage' (source + window)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.DISPUTE_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Dispute & chargeback review", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.DISPUTE_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified dispute-intelligence loop on free CFPB data. `setup.sql` already created a **`DISPUTE_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"what is our win rate by reason code, and where are we losing the most dollars?"*).

**Next:** point `DISPUTE_FACT` at your own chargeback ledger to analyze your real book against the CFPB complaint benchmark. This verified-note pattern reuses across other payments analytics use cases.

**Try breaking the verifier:** add a made-up forecast (e.g., *"friendly fraud likely to rise 15% next quarter"*) to the draft and watch Step 5 flag it *Needs review*.